# Fase 2 · QLoRA supervisado con Gemma y CLINC150

Cuaderno experimental autocontenido. No importa `gemma_clinc` ni lee sus YAML.
Entrena sobre `train`, selecciona un adaptador con `val` y no consulta `test`.
Los valores `4/4/1` permiten una prueba breve; `1510/302/95` es el piloto.
Usa `None` en los tres límites para ejecutar el experimento completo.


## 1. Parámetros

Estos valores replican `configs/phase2_qlora_sft.yaml`. Se pueden cambiar aquí
directamente para experimentar. Requiere GPU NVIDIA y acceso a Gemma.


In [ ]:
import hashlib
import json
import platform
import random
import re
import subprocess
import time
import urllib.request
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from bitsandbytes.nn import Linear4bit
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from tqdm.auto import tqdm
from transformers import (
    AutoModelForMultimodalLM,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
seed = 42
train_limit, eval_limit, max_steps = 4, 4, 1
model_id = "google/gemma-4-E2B-it"
revision = "3e22461f65e89153144f8adb70e3b8c2cc9845a7"
max_new_tokens = 16
system_prompt = (
    "Classify the user request into one CLINC150 intent. Return only the lowercase "
    "label with underscores. Valid labels were demonstrated during training."
)
epochs = 3
learning_rate, weight_decay = 2e-4, 0.01
accumulation, max_grad_norm, warmup_ratio = 16, 1.0, 0.03
lora_rank, lora_alpha, lora_dropout = 8, 16, 0.05
target_modules = ("q_proj", "v_proj")
gradient_checkpointing = False
log_every, eval_every_epochs = 25, 1
if max_steps is not None and max_steps < 1:
    raise ValueError("max_steps debe ser positivo")
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
print("Proyecto:", root)

## 2. Datos y subconjuntos equilibrados

Se conserva el orden oficial de CLINC150. Cuando usamos límites, recorremos
las etiquetas por turnos. Así el piloto incluye clases diversas.


In [ ]:
source_url = "https://raw.githubusercontent.com/clinc/oos-eval/master/data/data_full.json"
dataset_path = root / "data/raw/clinc150_full.json"
if not dataset_path.exists():
    dataset_path.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(source_url, headers={"User-Agent": "gemma-clinc/0.1"})
    with urllib.request.urlopen(request, timeout=60) as response:
        dataset_path.write_bytes(response.read())
with dataset_path.open(encoding="utf-8") as file:
    dataset = json.load(file)


def examples(split):
    rows = list(dataset[split]) + list(dataset[f"oos_{split}"])
    return [
        {
            "id": f"{split}-{i:05d}",
            "text": text,
            "label": "out_of_scope" if label == "oos" else label,
        }
        for i, (text, label) in enumerate(rows)
    ]


def balanced_subset(rows, limit):
    if limit is None:
        return rows
    if limit < 1:
        raise ValueError("El límite debe ser positivo")
    if limit >= len(rows):
        return rows
    buckets = {}
    for row in rows:
        buckets.setdefault(row["label"], []).append(row)
    offsets = dict.fromkeys(buckets, 0)
    selected = []
    while len(selected) < limit:
        for label, bucket in buckets.items():
            i = offsets[label]
            if i < len(bucket):
                selected.append(bucket[i])
                offsets[label] += 1
                if len(selected) == limit:
                    break
    return selected


train_all = examples("train")
train_rows = balanced_subset(train_all, train_limit)
val_rows = balanced_subset(examples("val"), eval_limit)
labels = sorted({row["label"] for row in train_all if row["label"] != "out_of_scope"})
labels.append("out_of_scope")
print(f"Train: {len(train_rows)}; validación: {len(val_rows)}; etiquetas: {len(labels)}")
print(train_rows[0])

## 3. Conversación supervisada

El usuario escribe la consulta original; la respuesta correcta es solo el
nombre de la intención. La lista completa de etiquetas no se repite aquí.


In [ ]:
def messages_for(example, include_answer=False):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": example["text"]},
    ]
    if include_answer:
        messages.append({"role": "assistant", "content": example["label"]})
    return messages


print(messages_for(train_rows[0], include_answer=True))

## 4. Gemma cuantizado y LoRA

NF4 comprime la base. La tabla grande de embeddings por capa se consulta en
CPU. Congelamos los pesos originales y entrenamos LoRA solo en `q_proj` y
`v_proj` de las capas de lenguaje.


In [ ]:
device_map = {"": 0, "model.language_model.embed_tokens_per_layer": "cpu"}
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)
processor = AutoProcessor.from_pretrained(model_id, revision=revision)
model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    revision=revision,
    device_map=device_map,
    dtype=torch.float16,
    quantization_config=quantization,
)
embedding = model.model.language_model.embed_tokens_per_layer
hook = getattr(embedding, "_hf_hook", None)
hooks = getattr(hook, "hooks", (hook,))
weights_map = next(
    (
        h.weights_map
        for h in hooks
        if getattr(h, "offload", False) and getattr(h, "weights_map", None) is not None
    ),
    None,
)
if weights_map is None:
    raise RuntimeError("No se encontró la tabla de embeddings descargada en CPU")
cpu_weight = weights_map["weight"].detach()
if cpu_weight.device.type != "cpu":
    raise RuntimeError("La tabla de embeddings no está en CPU")


class CpuPerLayerEmbedding(nn.Module):
    def __init__(self, weight, padding_idx, scale):
        super().__init__()
        self.weight = weight  # Congelado; no entra en el adaptador.
        self.padding_idx = padding_idx
        self.scale = scale

    def forward(self, input_ids):
        rows = F.embedding(input_ids.cpu(), self.weight, self.padding_idx)
        return (rows * self.scale).to(input_ids.device)


model.model.language_model.embed_tokens_per_layer = CpuPerLayerEmbedding(
    cpu_weight,
    embedding.padding_idx,
    embedding.scalar_embed_scale,
)
model.model.vision_tower = None
model.model.audio_tower = None
model.model.embed_vision = None
model.model.embed_audio = None
model.eval()

target_names = [
    name
    for name, layer in model.named_modules()
    if isinstance(layer, Linear4bit)
    and ".language_model.layers." in name
    and name.endswith(target_modules)
]
if not target_names:
    raise ValueError("No se encontraron proyecciones LoRA")
for parameter in model.parameters():
    parameter.requires_grad = False
text_config = (
    model.config.get_text_config() if hasattr(model.config, "get_text_config") else model.config
)
text_config.use_cache = False
if gradient_checkpointing:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
model = get_peft_model(
    model,
    LoraConfig(
        r=lora_rank,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_names,
        bias="none",
        task_type="CAUSAL_LM",
        revision=revision,
    ),
)
trainable = [p for p in model.parameters() if p.requires_grad]
trainable_parameters = sum(p.numel() for p in trainable)
if not trainable:
    raise RuntimeError("LoRA no produjo parámetros entrenables")
print(f"{len(target_names)} proyecciones; {trainable_parameters} parámetros entrenables")

## 5. Pérdida solo sobre la respuesta

Tokenizamos el prompt sin respuesta y la conversación completa. El primero
debe ser prefijo exacto de la segunda. Solo la etiqueta y el fin de turno
aportan pérdida; los logits del contexto ni siquiera se proyectan al vocabulario.


In [ ]:
def encode_training_example(example):
    prompt = processor.apply_chat_template(
        messages_for(example),
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    )
    full = processor.apply_chat_template(
        messages_for(example, include_answer=True),
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=False,
        enable_thinking=False,
    )
    prompt_ids = prompt["input_ids"][0].tolist()
    full_ids = full["input_ids"][0].tolist()
    if full_ids[: len(prompt_ids)] != prompt_ids or len(full_ids) <= len(prompt_ids):
        raise ValueError("La respuesta no sigue al prefijo del prompt")
    start, end = len(prompt_ids), len(full_ids)
    full = full.to(model.device)
    targets = full["input_ids"][:, start:end]
    logit_positions = torch.arange(start - 1, end - 1, device=model.device)
    return full, targets, logit_positions


batch, targets, positions = encode_training_example(train_rows[0])
print("Longitud total:", batch["input_ids"].shape[-1])
print("Tokens objetivo:", targets.tolist(), "| posiciones de logits:", positions.tolist())

## 6. Parser y métricas de validación

Guardamos tanto el texto bruto como la etiqueta interpretada. Las salidas
desconocidas o ambiguas cuentan como errores. Macro-F1 selecciona el adaptador.


In [ ]:
def parse_label(raw_output):
    normalized = raw_output.strip().lower().replace("-", "_").replace(" ", "_")
    normalized = normalized.strip("`'\".,:;!()[]{}")
    if normalized in labels:
        return normalized
    wrapped = re.search(r"(?:intent|label)_*[:=]_*([a-z0-9_]+)", normalized)
    if wrapped and wrapped.group(1) in labels:
        return wrapped.group(1)
    found = {
        label
        for label in labels
        if re.search(rf"(?<![a-z0-9_]){re.escape(label)}(?![a-z0-9_])", normalized)
    }
    return next(iter(found)) if len(found) == 1 else None


def measure(y_true, y_pred):
    predictions = [p if p is not None else "__invalid__" for p in y_pred]
    in_scope = [i for i, truth in enumerate(y_true) if truth != "out_of_scope"]
    oos_true = [truth == "out_of_scope" for truth in y_true]
    oos_pred = [pred == "out_of_scope" for pred in predictions]
    precision, recall, oos_f1, _ = precision_recall_fscore_support(
        oos_true,
        oos_pred,
        average="binary",
        zero_division=0,
    )
    confusions = Counter(
        (truth, pred) for truth, pred in zip(y_true, predictions, strict=True) if truth != pred
    )
    return {
        "n_examples": len(y_true),
        "accuracy": accuracy_score(y_true, predictions),
        "macro_f1": f1_score(
            y_true,
            predictions,
            labels=sorted(set(y_true)),
            average="macro",
            zero_division=0,
        ),
        "in_scope_accuracy": accuracy_score(
            [y_true[i] for i in in_scope],
            [predictions[i] for i in in_scope],
        ),
        "oos_precision": precision,
        "oos_recall": recall,
        "oos_f1": oos_f1,
        "invalid_output_rate": predictions.count("__invalid__") / len(predictions),
        "top_confusions": [
            {"true": truth, "predicted": pred, "count": count}
            for (truth, pred), count in confusions.most_common(20)
        ],
    }

## 7. Guardar artefactos y evaluar un checkpoint

Los archivos tienen el esquema del CLI: un JSONL para predicciones, un JSON
para métricas y un directorio PEFT por checkpoint. Cada ejecución crea una
carpeta nueva en `artifacts/phase2`.


In [ ]:
def write_json(path, value):
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n",
        encoding="utf-8",
    )


def append_jsonl(path, value):
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(value, ensure_ascii=False, default=str) + "\n")


try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=root,
        text=True,
        stderr=subprocess.DEVNULL,
    ).strip()
except (OSError, subprocess.CalledProcessError):
    git_commit = None
dataset_hash = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
run_dir = root / "artifacts/phase2" / f"phase2_qlora_sft_{run_id}"
run_dir.mkdir(parents=True, exist_ok=False)
history_path = run_dir / "training_history.jsonl"
write_json(run_dir / "label_vocabulary.json", labels)
config_snapshot = {
    "name": "phase2_qlora_sft",
    "seed": seed,
    "dataset": {
        "source_url": source_url,
        "cache_path": "data/raw/clinc150_full.json",
        "split": "val",
        "include_oos": True,
    },
    "model": {
        "id": model_id,
        "revision": revision,
        "quantization": "4bit",
        "dtype": "float16",
        "device_map": {"": 0},
        "offload_per_layer_embeddings": True,
        "text_only": True,
    },
    "prompt": {"system": system_prompt, "include_label_descriptions": False},
    "generation": {"max_new_tokens": max_new_tokens, "do_sample": False},
    "training": {
        "epochs": epochs,
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "gradient_accumulation_steps": accumulation,
        "max_grad_norm": max_grad_norm,
        "warmup_ratio": warmup_ratio,
        "lora_rank": lora_rank,
        "lora_alpha": lora_alpha,
        "lora_dropout": lora_dropout,
        "target_modules": target_modules,
        "gradient_checkpointing": gradient_checkpointing,
        "log_every": log_every,
        "eval_every_epochs": eval_every_epochs,
    },
    "output": {"root_dir": "artifacts/phase2", "checkpoint_every": 25},
}
write_json(
    run_dir / "run_metadata.json",
    {
        "run_id": run_id,
        "started_at": datetime.now(UTC).isoformat(),
        "git_commit": git_commit,
        "python": platform.python_version(),
        "platform": platform.platform(),
        "train_examples": len(train_rows),
        "validation_examples": len(val_rows),
        "train_limit": train_limit,
        "eval_limit": eval_limit,
        "max_steps": max_steps,
        "dataset_sha256": dataset_hash,
        "config": config_snapshot,
    },
)


def evaluate_validation(path):
    was_training = model.training
    model.eval()
    old_cache = text_config.use_cache
    text_config.use_cache = True
    y_true, y_pred = [], []
    try:
        for example in tqdm(val_rows, desc="Validation", unit="example"):
            inputs = processor.apply_chat_template(
                messages_for(example),
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
                add_generation_prompt=True,
                enable_thinking=False,
            ).to(model.device)
            prompt_length = inputs["input_ids"].shape[-1]
            with torch.no_grad():
                generated = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                )
            raw_output = processor.decode(
                generated[0][prompt_length:],
                skip_special_tokens=True,
            )
            prediction = parse_label(raw_output)
            append_jsonl(
                path,
                {
                    "id": example["id"],
                    "text": example["text"],
                    "expected": example["label"],
                    "predicted": prediction,
                    "raw_output": raw_output,
                    "correct": prediction == example["label"],
                },
            )
            y_true.append(example["label"])
            y_pred.append(prediction)
    finally:
        text_config.use_cache = old_cache
        model.train(was_training)
    return measure(y_true, y_pred)

## 8. Entrenar y elegir el mejor adaptador

Microbatch 1; acumulamos gradientes y actualizamos AdamW cada 16 ejemplos.
El último grupo de cada época usa su tamaño real. Guardamos un adaptador por
época y conservamos el primero que logra el mejor macro-F1 en validación.


In [ ]:
optimizer = torch.optim.AdamW(trainable, lr=learning_rate, weight_decay=weight_decay)
planned_steps = ((len(train_rows) + accumulation - 1) // accumulation) * epochs
total_steps = min(planned_steps, max_steps) if max_steps is not None else planned_steps
warmup_steps = round(total_steps * warmup_ratio)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
optimizer.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()
global_step = 0
best_macro_f1 = -1.0
best_adapter = None
stop_training = False
started = time.perf_counter()

for epoch in range(1, epochs + 1):
    order = list(range(len(train_rows)))
    random.Random(seed + epoch).shuffle(order)
    epoch_loss = 0.0
    epoch_examples = 0
    model.train()
    progress = tqdm(order, desc=f"Train epoch {epoch}", unit="example")
    for position, index in enumerate(progress, start=1):
        batch, targets, logit_positions = encode_training_example(train_rows[index])
        result = model(**batch, logits_to_keep=logit_positions, use_cache=False)
        loss = F.cross_entropy(
            result.logits.float().reshape(-1, result.logits.shape[-1]),
            targets.reshape(-1),
        )
        group_start = ((position - 1) // accumulation) * accumulation
        group_size = min(accumulation, len(order) - group_start)
        (loss / group_size).backward()
        epoch_loss += float(loss.detach())
        epoch_examples += 1

        if position % accumulation == 0 or position == len(order):
            learning_rate_used = optimizer.param_groups[0]["lr"]
            torch.nn.utils.clip_grad_norm_(trainable, max_grad_norm)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            if global_step % log_every == 0 or global_step == 1:
                record = {
                    "event": "optimizer_step",
                    "epoch": epoch,
                    "global_step": global_step,
                    "mean_epoch_loss": epoch_loss / epoch_examples,
                    "learning_rate_used": learning_rate_used,
                    "next_learning_rate": scheduler.get_last_lr()[0],
                    "elapsed_seconds": time.perf_counter() - started,
                    "cuda_peak_allocated_mib": round(torch.cuda.max_memory_allocated() / 2**20, 1),
                    "cuda_peak_reserved_mib": round(torch.cuda.max_memory_reserved() / 2**20, 1),
                }
                append_jsonl(history_path, record)
                progress.set_postfix(loss=f"{record['mean_epoch_loss']:.4f}")
            if max_steps is not None and global_step >= max_steps:
                stop_training = True
                break

    adapter_dir = run_dir / f"adapter_epoch_{epoch:03d}_step_{global_step:06d}"
    model.save_pretrained(adapter_dir)
    epoch_record = {
        "event": "epoch_end",
        "epoch": epoch,
        "global_step": global_step,
        "mean_loss": epoch_loss / max(epoch_examples, 1),
        "adapter": str(adapter_dir),
    }
    if epoch % eval_every_epochs == 0 or stop_training:
        predictions_path = run_dir / f"validation_epoch_{epoch:03d}.jsonl"
        validation_metrics = evaluate_validation(predictions_path)
        write_json(run_dir / f"validation_metrics_epoch_{epoch:03d}.json", validation_metrics)
        epoch_record["validation"] = validation_metrics
        if validation_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = validation_metrics["macro_f1"]
            best_adapter = str(adapter_dir)
    append_jsonl(history_path, epoch_record)
    if stop_training:
        break

summary = {
    "completed_at": datetime.now(UTC).isoformat(),
    "optimizer_steps": global_step,
    "trainable_parameters": trainable_parameters,
    "best_macro_f1": best_macro_f1,
    "best_adapter": best_adapter,
    "cuda_peak_allocated_mib": round(torch.cuda.max_memory_allocated() / 2**20, 1),
    "cuda_peak_reserved_mib": round(torch.cuda.max_memory_reserved() / 2**20, 1),
    "elapsed_seconds": time.perf_counter() - started,
}
write_json(run_dir / "training_summary.json", summary)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("Artefactos:", run_dir)

`training_summary.json` contiene `best_adapter`. Es un adaptador PEFT normal,
recargable por el CLI. Evaluar `test` solo cuando el experimento esté fijado.
